# Data Ingestion/Preprocessing
Utilize Last Observation Carried Forward (LOCF) for tracking spread where last known price of an asset remains valid until a new quote overwrites it.


In [ ]:
import pandas as pd
import numpy as np
from preprocessing import reconstruct_continuous_book

# ---------------------------------------------------------
# 1. DATA INGESTION & SYNCHRONIZATION
# ---------------------------------------------------------
df_perp = reconstruct_continuous_book("BTCPERP")
df_spot = reconstruct_continuous_book("BTCUSDT")

df_perp = df_perp.rename(columns={'best_bid': 'perp_bid', 'best_ask': 'perp_ask'})
df_spot = df_spot.rename(columns={'best_bid': 'spot_bid', 'best_ask': 'spot_ask'})

print("Aligning asynchronous data streams...")
df_arb = pd.merge_asof(
    df_perp[['time', 'perp_bid', 'perp_ask']], 
    df_spot[['time', 'spot_bid', 'spot_ask']], 
    on='time', 
    direction='backward'
)
df_arb = df_arb.dropna().copy()
df_arb

[BTCPERP] Fetching raw LOB events from database...
[BTCPERP] Reconstructing book from 503915 events...
[BTCPERP] Successfully generated 97116 continuous BBO states.

[BTCUSDT] Fetching raw LOB events from database...
[BTCUSDT] Reconstructing book from 920292 events...
[BTCUSDT] Successfully generated 124085 continuous BBO states.

Aligning asynchronous data streams...


,time,perp_bid,perp_ask,spot_bid,spot_ask
9,2026-07-27 04:53:02.634000+00:00,65242.1,65242.2,65279.8,65296.7
10,2026-07-27 04:53:02.654000+00:00,65242.1,65242.2,65279.8,65295.4
11,2026-07-27 04:53:02.695000+00:00,65242.1,65242.2,65282.1,65293.8
12,2026-07-27 04:53:02.714000+00:00,65242.1,65242.2,65282.1,65293.8
13,2026-07-27 04:53:02.734000+00:00,65242.1,65242.2,65282.1,65293.8
...,...,...,...,...,...
97111,2026-07-28 17:42:44.474000+00:00,63523.1,63523.2,63574.3,63574.4
97112,2026-07-28 17:42:44.514000+00:00,63523.1,63523.2,63574.3,63574.4
97113,2026-07-28 17:42:44.534000+00:00,63523.1,63523.2,63574.3,63574.4
97114,2026-07-28 17:42:44.540000+00:00,63523.1,63523.2,63574.3,63574.4


# QR1: Arbitraging Threshold, When is Slippage Too Expensive? Exploring Arbitrage Decay.
Solve this by calculating zscore levels relative to price level then making this to a histogram curve to investigate and isolate spread threshold.

In [7]:
import numpy as np
import pandas as pd
from preprocessing import reconstruct_continuous_book

# =====================================================================
# 1. DATA INGESTION, SYNCHRONIZATION & HYGIENE
# =====================================================================
def load_and_align_books():
    print("Loading continuous order book feeds...")
    df_perp = reconstruct_continuous_book("BTCPERP")
    df_spot = reconstruct_continuous_book("BTCUSDT")

    # Rename quote columns and preserve raw timestamps for staleness tracking
    df_perp = df_perp.rename(columns={'best_bid': 'perp_bid', 'best_ask': 'perp_ask'})
    df_spot = df_spot.rename(columns={'best_bid': 'spot_bid', 'best_ask': 'spot_ask'})
    
    df_perp['perp_time'] = df_perp['time']
    df_spot['spot_time'] = df_spot['time']

    print("Aligning asynchronous data streams...")
    df_arb = pd.merge_asof(
        df_perp[['time', 'perp_time', 'perp_bid', 'perp_ask']], 
        df_spot[['time', 'spot_time', 'spot_bid', 'spot_ask']], 
        on='time', 
        direction='backward'
    )
    
    df_arb = df_arb.dropna().reset_index(drop=True)
    df_arb['time'] = pd.to_datetime(df_arb['time'])
    df_arb['perp_time'] = pd.to_datetime(df_arb['perp_time'])
    df_arb['spot_time'] = pd.to_datetime(df_arb['spot_time'])
    
    return df_arb

def prepare_signals(df, window=1000, max_staleness_ms=100, min_vol_usd=1.0):
    df = df.copy()
    
    # 1. Feed Staleness Diagnostic
    df['feed_latency_ms'] = (df['perp_time'] - df['spot_time']).abs().dt.total_seconds() * 1000.0
    df['is_stale'] = df['feed_latency_ms'] > max_staleness_ms
    
    # 2. Mid Prices & Signal Spread
    df['perp_mid'] = (df['perp_bid'] + df['perp_ask']) / 2.0
    df['spot_mid'] = (df['spot_bid'] + df['spot_ask']) / 2.0
    df['mid_spread'] = df['perp_mid'] - df['spot_mid']
    
    # 3. Executable Crossing Spreads (Realizable Market Orders)
    # Excursion >= +1: Sell Perp @ Bid, Buy Spot @ Ask
    df['exec_short_spread'] = df['perp_bid'] - df['spot_ask'] 
    # Excursion <= -1: Buy Perp @ Ask, Sell Spot @ Bid
    df['exec_long_spread'] = df['perp_ask'] - df['spot_bid']  
    
    # 4. Robust Rolling Z-score (Median & MAD with Volatility Floor)
    rolling_med = df['mid_spread'].rolling(window=window, min_periods=window//2).median()
    rolling_mad = (df['mid_spread'] - rolling_med).abs().rolling(window=window, min_periods=window//2).median()
    
    scale = np.maximum(rolling_mad * 1.4826, min_vol_usd)
    df['z_score'] = (df['mid_spread'] - rolling_med) / scale
    
    # Zero out signal if quote staleness exceeds threshold
    df.loc[df['is_stale'], 'z_score'] = 0.0
    return df

# =====================================================================
# 2. MICROSTRUCTURE BACKTEST ENGINE WITH LATENCY REPRICING
# =====================================================================
def run_latency_backtest(df, latencies_ms=[0, 50, 100, 200, 300, 500, 1000], fee_hurdle_usd=2.0):
    events = []
    in_excursion = False
    
    timestamps = df['time'].values
    z_scores = df['z_score'].values
    exec_short = df['exec_short_spread'].values
    exec_long = df['exec_long_spread'].values
    n = len(df)
    
    i = 0
    while i < n:
        current_z = z_scores[i]
        
        if not in_excursion and abs(current_z) >= 1.0:
            in_excursion = True
            signal_time = timestamps[i]
            entry_z = current_z
            peak_z = abs(current_z)
            excursion_type = 1 if current_z >= 1.0 else -1  # 1: Short Perp/Long Spot, -1: Long Perp/Short Spot
            
            # Reprice entry fills at signal_time + latency horizon
            fill_short_spreads = {}
            fill_long_spreads = {}
            
            for lat in latencies_ms:
                target_time = signal_time + np.timedelta64(lat, 'ms')
                fill_idx = np.searchsorted(timestamps, target_time)
                if fill_idx < n:
                    fill_short_spreads[lat] = exec_short[fill_idx]
                    fill_long_spreads[lat] = exec_long[fill_idx]
                else:
                    fill_short_spreads[lat] = exec_short[-1]
                    fill_long_spreads[lat] = exec_long[-1]
            
            i += 1
            continue
            
        if in_excursion:
            if abs(current_z) > peak_z:
                peak_z = abs(current_z)
                
            reverted = (excursion_type == 1 and current_z <= 0) or \
                       (excursion_type == -1 and current_z >= 0)
            
            if reverted:
                revert_time = timestamps[i]
                ttr_ms = (revert_time - signal_time) / np.timedelta64(1, 'ms')
                
                # Unwind fills at reversion (Exit)
                exit_short_pay = exec_long[i]   # To exit Short Perp/Long Spot, buy Perp @ Ask, sell Spot @ Bid
                exit_long_pay = exec_short[i]   # To exit Long Perp/Short Spot, sell Perp @ Bid, buy Spot @ Ask
                
                event_data = {
                    'peak_z_score': peak_z,
                    'entry_z': entry_z,
                    'ttr_ms': ttr_ms,
                    'excursion_type': excursion_type
                }
                
                # PnL Calculation across Latencies
                for lat in latencies_ms:
                    if excursion_type == 1:
                        # Entered at exec_short, exited at exit_short_pay
                        gross_pnl = fill_short_spreads[lat] - exit_short_pay
                    else:
                        # Entered at exec_long, exited at exit_long_pay
                        gross_pnl = exit_long_pay - fill_long_spreads[lat]
                        
                    net_pnl = gross_pnl - fee_hurdle_usd
                    event_data[f'net_pnl_{lat}ms'] = net_pnl
                    event_data[f'is_prof_{lat}ms'] = net_pnl > 0
                
                events.append(event_data)
                in_excursion = False
                
        i += 1

    return pd.DataFrame(events)

# =====================================================================
# 3. REPORT GENERATOR (TTR, PROFITABILITY & DECAY)
# =====================================================================
def bootstrap_ci(series, n_boot=1000, ci=95):
    if len(series) == 0:
        return (0.0, 0.0)
    boot_means = [np.random.choice(series, size=len(series), replace=True).mean() for _ in range(n_boot)]
    return np.percentile(boot_means, (100 - ci) / 2), np.percentile(boot_means, 100 - (100 - ci) / 2)

def generate_quant_reports(df_events, target_latency_ms=200):
    if len(df_events) == 0:
        print("No mean-reversion excursions identified.")
        return

    bins = [1.0, 2.0, 3.0, 4.0, np.inf]
    labels = ['1.0 - 2.0', '2.0 - 3.0', '3.0 - 4.0', '4.0+']
    df_events['z_tier'] = pd.cut(df_events['peak_z_score'], bins=bins, labels=labels, right=False)

    print("\n" + "="*65)
    print("=== 1. TTR (REVERSION DURATION) DISTRIBUTION (ms) ===")
    print("="*65)
    ttr_stats = df_events.groupby('z_tier', observed=False)['ttr_ms'].agg(
        Count='count',
        p5=lambda x: np.percentile(x, 5) if len(x) > 0 else 0,
        p25=lambda x: np.percentile(x, 25) if len(x) > 0 else 0,
        Median='median',
        p75=lambda x: np.percentile(x, 75) if len(x) > 0 else 0,
        p95=lambda x: np.percentile(x, 95) if len(x) > 0 else 0,
        Mean='mean'
    ).reset_index()
    print(ttr_stats.to_string(index=False))

    print("\n" + "="*65)
    print(f"=== 2. PROFITABILITY MATRIX AT {target_latency_ms}ms LATENCY ===")
    print("="*65)
    records = []
    for tier, group in df_events.groupby('z_tier', observed=False):
        pnl_col = f'net_pnl_{target_latency_ms}ms'
        prof_col = f'is_prof_{target_latency_ms}ms'
        
        count = len(group)
        if count > 0:
            win_rate = group[prof_col].mean() * 100
            l_ci, u_ci = bootstrap_ci(group[prof_col].astype(int).values)
            avg_pnl = group[pnl_col].mean()
        else:
            win_rate, l_ci, u_ci, avg_pnl = 0, 0, 0, 0
            
        records.append({
            'Z_Tier': tier,
            'Count': count,
            'Avg_Net_PnL': f"${avg_pnl:.2f}",
            'Win_Rate': f"{win_rate:.1f}%",
            'Bootstrapped_95%_CI': f"[{l_ci*100:.1f}%, {u_ci*100:.1f}%]"
        })
    print(pd.DataFrame(records).to_string(index=False))

    print("\n" + "="*65)
    print("=== 3. ALPHA DECAY CURVE (AVG NET PNL BY LATENCY) ===")
    print("="*65)
    lat_cols = [c for c in df_events.columns if c.startswith('net_pnl_')]
    decay_summary = {}
    for col in lat_cols:
        lat_val = col.replace('net_pnl_', '')
        decay_summary[lat_val] = df_events[col].mean()
    
    decay_df = pd.DataFrame(list(decay_summary.items()), columns=['Latency Horizon', 'Avg Net PnL ($)'])
    print(decay_df.to_string(index=False))

# =====================================================================
# 4. EXECUTION
# =====================================================================
if __name__ == "__main__":
    # 1. Ingest and Synchronize
    df_arb = load_and_align_books()
    
    # 2. Apply Robust Z-Score and Feed Staleness Filters
    df_signals = prepare_signals(df_arb, window=1000, max_staleness_ms=100, min_vol_usd=1.0)
    
    # 3. Simulate Market Orders with Realistic Execution Latencies
    df_events = run_latency_backtest(
        df_signals, 
        latencies_ms=[0, 50, 100, 200, 300, 500, 1000], 
        fee_hurdle_usd=2.0
    )
    
    # 4. Output Quantitative Report
    generate_quant_reports(df_events, target_latency_ms=200)

Loading continuous order book feeds...
[BTCPERP] Fetching raw LOB events from database...
[BTCPERP] Reconstructing book from 534888 events...
[BTCPERP] Successfully generated 102674 continuous BBO states.

[BTCUSDT] Fetching raw LOB events from database...
[BTCUSDT] Reconstructing book from 978919 events...
[BTCUSDT] Successfully generated 130983 continuous BBO states.

Aligning asynchronous data streams...

=== 1. TTR (REVERSION DURATION) DISTRIBUTION (ms) ===
   z_tier  Count   p5   p25  Median    p75    p95          Mean
1.0 - 2.0    511 80.0 320.0   800.0 1939.5 5677.5 202143.702544
2.0 - 3.0    191 60.5 301.0   805.0 2749.0 7947.0 127797.984293
3.0 - 4.0     97 59.8 440.0  1160.0 3640.0 7221.0   2971.525773
     4.0+    237 96.8 281.0   880.0 2444.0 5967.8   1745.987342

=== 2. PROFITABILITY MATRIX AT 200ms LATENCY ===
   Z_Tier  Count Avg_Net_PnL Win_Rate Bootstrapped_95%_CI
1.0 - 2.0    511      $11.59    67.5%      [63.8%, 71.2%]
2.0 - 3.0    191       $6.13    64.4%      [57.6

In [3]:
# ---------------------------------------------------------
# 2. STATARB EVENT-DRIVEN STATE MACHINE
# ---------------------------------------------------------
print("Extracting statistical arbitrage excursions...")

# Round-trip execution fee hurdle across both legs (e.g., $5.00 total)
FEE_HURDLE_USD = 50.00

events = []
in_excursion = False
start_time = None
entry_spread = 0.0
entry_z = 0.0
peak_z = 0.0
excursion_type = 0  # +1 for Upper (Short Spread), -1 for Lower (Long Spread)

for row in df_arb.itertuples():
    current_z = row.z_score
    current_time = row.time
    current_spread = row.spread
    
    # TRIGGER: Absolute Z-Score >= 1.0 (Enters Dislocation)
    if not in_excursion and abs(current_z) >= 1.0:
        in_excursion = True
        start_time = current_time
        entry_spread = current_spread
        entry_z = current_z
        peak_z = abs(current_z)
        excursion_type = 1 if current_z >= 1.0 else -1
        
    elif in_excursion:
        # Track maximum Z-score intensity reached during this event
        if abs(current_z) > peak_z:
            peak_z = abs(current_z)
            
        # EXIT CONDITION: Mean Reversion achieved (Z crosses 0)
        reverted = (excursion_type == 1 and current_z <= 0) or \
                   (excursion_type == -1 and current_z >= 0)
        
        if reverted:
            end_time = current_time
            exit_spread = current_spread
            ttr_ms = (end_time - start_time).total_seconds() * 1000.0  # Duration in ms
            
            # Calculate Realized PnL from Reversion
            if excursion_type == 1:
                gross_pnl = entry_spread - exit_spread  # Short spread profit
            else:
                gross_pnl = exit_spread - entry_spread  # Long spread profit
                
            net_pnl = gross_pnl - FEE_HURDLE_USD
            
            events.append({
                'peak_z_score': peak_z,
                'entry_z': entry_z,
                'entry_spread': entry_spread,
                'exit_spread': exit_spread,
                'gross_pnl_usd': gross_pnl,
                'net_pnl_usd': net_pnl,
                'ttr_ms': ttr_ms,
                'is_profitable': net_pnl > 0
            })
            
            in_excursion = False

df_events = pd.DataFrame(events)

# ---------------------------------------------------------
# 3. STATISTICAL THRESHOLD MATRIX GENERATION
# ---------------------------------------------------------
if len(df_events) > 0:
    # Bin by the peak Z-score reached during the excursion
    bins = [1.0, 2.0, 3.0, 4.0, np.inf]
    labels = ['1.0 - 2.0', '2.0 - 3.0', '3.0 - 4.0', '4.0+']
    df_events['z_tier'] = pd.cut(df_events['peak_z_score'], bins=bins, labels=labels, right=False)

    quant_matrix = df_events.groupby('z_tier', observed=False).agg(
        Opportunity_Count=('peak_z_score', 'count'),
        Avg_Entry_Spread=('entry_spread', 'mean'),
        Avg_Gross_Profit_USD=('gross_pnl_usd', 'mean'),
        Win_Rate_Profitable=('is_profitable', 'mean'), 
        Avg_TTR_ms=('ttr_ms', 'mean')
    ).reset_index()

    # Formatting output for quant presentation
    quant_matrix['Win_Rate_Profitable'] = (quant_matrix['Win_Rate_Profitable'] * 100).map("{:.1f}%".format)
    quant_matrix['Avg_Entry_Spread'] = quant_matrix['Avg_Entry_Spread'].map("${:.2f}".format)
    quant_matrix['Avg_Gross_Profit_USD'] = quant_matrix['Avg_Gross_Profit_USD'].map("${:.2f}".format)
    quant_matrix['Avg_TTR_ms'] = quant_matrix['Avg_TTR_ms'].map("{:.1f}".format)

    print("\n=== STATISTICAL ARBITRAGE DECAY & PROFITABILITY MATRIX ===")
    print(quant_matrix.to_string(index=False))
else:
    print("No mean-reversion excursions found in this dataset.")

Extracting statistical arbitrage excursions...

=== STATISTICAL ARBITRAGE DECAY & PROFITABILITY MATRIX ===
   z_tier  Opportunity_Count Avg_Entry_Spread Avg_Gross_Profit_USD Win_Rate_Profitable Avg_TTR_ms
1.0 - 2.0                 43          $-42.70               $10.96                0.0%     7191.8
2.0 - 3.0                 25          $-59.13               $23.42                0.0%    11967.8
3.0 - 4.0                 20          $-59.46               $31.97               15.0%    17868.2
     4.0+                 99          $-71.41               $40.88               26.3%    21239.4


In [4]:
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Ensure 'time' column is parsed as datetime
df_arb['time'] = pd.to_datetime(df_arb['time'])

# Reset index to guarantee sequential 0..N integer positioning (removes time gaps)
df_plot = df_arb.reset_index(drop=True)
x_axis = df_plot.index

# Create a 2-panel interactive subplot stacked vertically sharing x-axis
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.08,
    subplot_titles=("Arbitrage Basis Spread ($)", "Z-Score & Trigger Thresholds")
)

# ---------------------------------------------------------
# TOP PANEL: Spot / Perp Basis Spread ($)
# ---------------------------------------------------------
fig.add_trace(
    go.Scatter(
        x=x_axis, 
        y=df_plot['spread'],
        mode='lines',
        name='Spread ($)',
        line=dict(color='#1f77b4', width=1.2),
        customdata=df_plot['time'],
        hovertemplate='<b>Time</b>: %{customdata|%Y-%m-%d %H:%M:%S}<br><b>Spread</b>: $%{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

# ---------------------------------------------------------
# BOTTOM PANEL: Z-Score with Trigger Thresholds
# ---------------------------------------------------------
fig.add_trace(
    go.Scatter(
        x=x_axis, 
        y=df_plot['z_score'],
        mode='lines',
        name='Z-Score',
        line=dict(color='#2ca02c', width=1.2),
        customdata=df_plot['time'],
        hovertemplate='<b>Time</b>: %{customdata|%Y-%m-%d %H:%M:%S}<br><b>Z-Score</b>: %{y:.2f}<extra></extra>'
    ),
    row=2, col=1
)

# Add Reference Threshold Lines to Bottom Panel
fig.add_hline(y=0.0, line=dict(color='black', width=1.2), row=2, col=1, annotation_text="Mean (Z=0)")
fig.add_hline(y=1.0, line=dict(color='orange', width=1, dash='dash'), row=2, col=1)
fig.add_hline(y=-1.0, line=dict(color='orange', width=1, dash='dash'), row=2, col=1)
fig.add_hline(y=2.0, line=dict(color='red', width=1.2, dash='dot'), row=2, col=1, annotation_text="High Dislocation (|Z|=2)")
fig.add_hline(y=-2.0, line=dict(color='red', width=1.2, dash='dot'), row=2, col=1)

# ---------------------------------------------------------
# LAYOUT & CUSTOM X-AXIS TICK FORMATTING (GAPLESS)
# ---------------------------------------------------------
# Generate evenly spaced tick positions for time labels
num_ticks = 8
tick_positions = [int(x) for x in np.linspace(0, len(df_plot) - 1, num_ticks)]
tick_labels = df_plot['time'].iloc[tick_positions].dt.strftime('%H:%M:%S').tolist()

fig.update_layout(
    title_text="<b>Interactive Arbitrage Basis Spread & Z-Score Analysis</b>",
    template="plotly_white",
    height=800,
    hovermode='x unified',
    showlegend=False
)

# Map clean time labels to sequential index ticks
fig.update_xaxes(
    tickvals=tick_positions,
    ticktext=tick_labels,
    title_text="Time (UTC)",
    row=2, col=1
)
fig.update_yaxes(title_text="Spread ($)", row=1, col=1)
fig.update_yaxes(title_text="Z-Score", row=2, col=1)

# Display the interactive plot (renders natively in Jupyter Notebooks/Lab)
fig.show()

# Preventing Lookahead Bias
Separate target from features with very clear definition.

In [ ]:
from features import generate_features, generate_target

# Load raw data
raw_data = df_arb

# Build features and targets separately
X = generate_features(raw_data)
y = generate_target(raw_data)

# Drop rows where NaNs were created by shifting
model_data = pd.concat([X, y], axis=1).dropna()